In [1]:
import polars as pl
from datetime import datetime
import duckdb

In [ ]:
# Bronze landing stage
filename = "results.csv"
with open('data/results.csv', mode='r') as file:
    data = pl.scan_csv(file, has_header = True, separator = ",")

# Add lineage metadata
df = data.with_columns(
    pl.lit((datetime.now().strftime("%Y-%m-%d"))).alias("bronze_landing_date_ingested"),
    pl.lit(("https://www.kaggle.com/datasets/lylebegbie/international-rugby-union-results-from-18712022")).alias("bronze_source"),
    pl.lit("#").alias("pipeline_id"),
    pl.lit(1).alias("run_id")
)

# Convert to parquet for storage 
df.sink_parquet("data/bronze_landing_data.parquet")

In [ ]:
# Bronze landing zone
df = pl.read_parquet("data/bronze_landing_data.parquet")

# check for schema drift
source_columns = set(['date','home_team','away_team','home_score','away_score','competition','stadium','city','country','neutral','world_cup'])
bronze_metadata_columns = {'run_id', 'pipeline_id', 'bronze_landing_date_ingested', 'bronze_source'}

land_dataset_columns = set(df.columns)

column_diff = land_dataset_columns - source_columns - bronze_metadata_columns

if column_diff:
	print("There is an extra column(s) in your data")

# Rename the columns
df = df.rename(({"date": "match_date"}))

# Change data types
df = df.cast({"home_team": pl.String, 
			"away_team": pl.String, 
			"home_score": pl.Int64, 
			"away_score": pl.Int64,
			"competition": pl.String,
			"stadium": pl.String,
			"city": pl.String, 
            "country": pl.String,
            "neutral": pl.Boolean,
            "world_cup": pl.Boolean,
            "bronze_source": pl.String})

# Add lineage metadata
df = df.with_columns(
    pl.lit((datetime.now().strftime("%Y-%m-%d"))).alias("bronze_raw_date_ingested")
    )

df.write_parquet("data/bronze_raw_data.parquet")

In [ ]:
# Silver Source alinged
df = pl.read_parquet("data/bronze_raw_data.parquet")

# Add the source hash key and business hash keys 
# keep the seed key the same
df = df.with_columns(
    pl.struct(['match_date','home_team','away_team','home_score','away_score','competition','stadium','city','country','neutral','world_cup']).hash(seed=0).alias("silver_source_hash")
)

# Dedup on the source hash
df_de_dup = df.unique(pl.col("silver_source_hash"))

# Clean the string columns
string_cols = ["home_team", "away_team", "competition", "stadium", "city", "country", ]

df_de_dup = df_de_dup.with_columns(
    pl.col(c).str.strip_chars().str.to_titlecase() for c in string_cols
)

# Handle Null values
required_cols = ["match_date", "home_team", "away_team", "home_score", "away_score", "competition"] # including competition as it 

violations = df_de_dup.filter(
    pl.any_horizontal([pl.col(c).is_null() for c in required_cols])
)

violation_expr = pl.any_horizontal([pl.col(c).is_null() for c in required_cols])

violations = df_de_dup.filter(violation_expr)
clean = df_de_dup.filter(~violation_expr)

# Null violations are quarintined files 
violations.write_parquet("data/silver_quarintine_kaggle.parquet")

# Value Standerdizations: Country, Stadium, teams, Competition, 

silver_reference_team = pl.DataFrame({"team":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
silver_reference_country = pl.DataFrame({"country":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
# Competition needs to be thought through more as they include groups stages in world cups etc 
silver_reference_stadium = pl.DataFrame({"stadium": ["Stade De France","Stadium Australia","Rectory Field","Ballymore Stadium","Inverleith","Stadio Plebiscito","Estadio José Fierro","Stade Pierre-Mauroy","Perth Stadium","Hamilton Crescent","Estadio Ricardo Etcheverry","Stadio Olimpico","St. Helen'S","Jade Stadium","Nelson Mandela Bay Stadium","Stadio Luigi Ferraris","Whalley Range","Olympic Park Stadium","Estadio Gimnasia Y Esgrima De Buenos Aires","Adelaide Oval","Perth Oval","Estadio Gigante De Arroyito","Stade Municipal","Rotorua Int. Stadium","Cardigan Fields","Meanwood Road","North Queensland Stadium","Estadio Centenario","Estadio José Amalfitani","Estadio Raúl Conti","Stade Du Moulias","Kingsmead Cricket Ground","Estadio Brigadier General Estanislao López","Concord Oval","Aviva Stadium","Sydney Cricket Ground","Westpac Stadium","Lancaster Park","Balmoral Showgrounds","Wembley Stadium","Stadio Olimpico Di Torino","Loftus Versfeld","Stade Geoffroy-Guichard","West Of Scotland F.C.","International Stadium Yokohama","Suncorp Stadium","Stadio Comunale Mario Battaglini","Ballymore","Shizuoka Stadium Ecopa","Stadio Arturo Collana","Colonial Stadium","Sydney Sports Ground","Estadio Estanislao López","Parc Olympique","Brisbane Cricket Ground","Yves-Du-Manoir","Estadio B.G Estanislao López","Ami Stadium","Rugby League Park","Rfk Stadium","St George'S Park","Estadio San Juan Del Bicentenario","Westpactrust Stadium","St Helen'S","Athletic Park","Stade Marcel Michelin","Millennium Stadium","Canberra Stadium","Stadium Municipal","José Amalfitani Stadium","Kings Park Stadium","Ferro Carrill Oeste","Sydney Football Stadium","Kings Park","Optus Stadium","Soldier Field","Newlands","Colombes","Galpharm Stadium","Gare De La Croix Du Prince","Nissan Stadium","San Siro Stadium","Stade Armandie","Melbourne Cricket Ground","King'S Park","Grand Stade Lille Métropole","Stadio Renato Dall'Ara","St Helens","Welford Road Stadium","Powderhall Stadium","Parc Des Princes","Murrayfield Stadium","Lansdowne Road","Stade Yves-Du-Manoir","Ōita Stadium","Mcalpine Stadium","Melbourne Rec. Stadium","Rodney Parade","Estadio Etcheverry","Epru Stadium","Mclean Park","Estadio 23 De Agosto","Stadio Artemio Franchi","Stade De Gerland","Vodacom Park","Powderhall","Stade De La Mosson","Buffalo City Stadium","Estadio Mario Alberto Kempes","Regional Stadium","Trafalgar Park","Kingsmead","Estadio Madre De Ciudades","Ashton Gate","Fallowfield","Stade De La Meinau","Stade Mayol","Allianz Riviera","Stade Félix Bollaert","Royal Bafokeng Stadium","Richardson'S Field","Crystal Palace","Ellis Park","Carisbrook","Stade Vélodrome","Hong Kong Stadium","Stadio Flaminio","Stade Maurice Trélut","Ormeau Cricket Ground","Vélez Sársfield","Murrayfield","Free State Stadium","Ferrocarril Stadium","Stade De La Beaujoire","Mardyke","Ravenhill Stadium","Mount Smart Stadium","Stadio Comunale Di Monigo","Croke Park","St James Park","Exhibition Ground","Epsom Showgrounds","Sky Stadium","Stade Marcel Saupin","Stade Lesdiguières","Gigante De Arroyito","Mcdonald Jones Stadium","Fc Oeste","Cbus Super Stadium","Upper Park","Kingsholm","Johann Van Riebeeck Stadium","Twickenham","Cardiff Arms Park","Eden Park","Ulster Cricket Ground","Docklands Stadium","Parc Y Scarlets","Stade Colombes","Estadio Padre Ernesto Martearena","Melbourne Rectangular Stadium","Stade Chaban-Delmas","Stade Des Ponts Jumeaux","Waikato Stadium","Stadio Euganeo","Olympic Stadium","Forsyth Barr Stadium","José Amalfitani","Hampden Park","Stradey Park","Ferrocaril Oeste","Thomond Park","Estadio Único","River Plate Stadium","Welford Road","Estadio Bicentenario","Pam Brink Stadium","Stadium Nord Lille Métropole","Telstra Dome","Stadio Marc'Antonio Bentegodi","Stadio Marassi","Stadio Mompiano","Twickenham Stadium","Raeburn Place","Loftus Versfeld Stadium","Malvinas Argentinas","Estadio José María Minella","Robina Stadium","Ellis Park Stadium","Absa Stadium","Boet Erasmus Stadium","Parc Lescure","Subiaco Oval","Newlands Stadium","Otago Stadium","Brisbane Exhibition Ground","Stadium Lille-Metropole","Tokyo Stadium","Crusaders Ground","Headingley","North Harbour Stadium","Tahuna Park","Arena Civica","Crown Flatt","Stadio Comunale Beltrametti","Rathmines","National Stadium","Lang Park","Estadio Monumental José Fierro","Yarrow Stadium","Estadio Malvinas Argentinas","Wellington Regional Stadium","Athletic Ground","Bankwest Stadium","The Oval","Fnb Stadium","Mbombela Stadium","Ravenhill","Padre Ernesto Martearena","Cape Town Stadium","Estadio Olímpico","Birkenhead Park","Bruce Stadium","Rugby Park","Springbok Park","Telstra Stadium","Stadio Xxv Aprile","Stadio Friuli","Old Trafford","St James' Park","Stade Pershing", "Commbank Stadium"]})



# Add lineage metadata

silver_metadata = clean.with_columns(
				pl.lit((datetime.now().strftime("%Y-%m-%dT%H:%M:%S"))).alias("silver_source_processed"),
)

silver_metadata.write_parquet("data/silver_pre_nf.parquet")

In [ ]:
# Silver 3nf
# Country
# load the data from the silver parquet file
df = pl.read_parquet("data/silver_pre_nf.parquet")

# Surviorship and column selection
df = df.unique(subset="country")
country_df = df.select((["country","run_id","pipeline_id","silver_source_processed","bronze_raw_date_ingested","bronze_landing_date_ingested","bronze_source"]))

con = duckdb.connect("data/silver_3nf.duckdb")
# UPSERT country
con.execute("""
USE silver;
            
MERGE INTO country AS tgt
USING country_df AS src
  ON tgt.country = src.country
WHEN MATCHED THEN UPDATE SET
    run_id                       = src.run_id,
    pipeline_id                  = src.pipeline_id,
    silver_source_processed        = CURRENT_DATE,
WHEN NOT MATCHED THEN INSERT (
    country, run_id, pipeline_id, silver_source_processed,
    bronze_raw_date_ingested, bronze_landing_date_ingested, bronze_source
) VALUES (
    src.country, src.run_id, src.pipeline_id, src.silver_source_processed,
    src.bronze_raw_date_ingested, src.bronze_landing_date_ingested, src.bronze_source
);
""")


In [ ]:
# Stadium
# load the data from the silver parquet file
df = pl.read_parquet("data/silver_pre_nf.parquet")
con = duckdb.connect("data/silver_3nf.duckdb")

# resolve references
country_lookup_df = con.execute("select country_id, country from silver.country").pl()

df = df.join(country_lookup_df, on="country", how="left")

# Surviorship and column selection
stadium_df = df.select((["stadium","country_id","run_id","pipeline_id","silver_source_processed","bronze_raw_date_ingested","bronze_landing_date_ingested","bronze_source"]))
stadium_df = df.unique(subset="stadium")
stadium_df = stadium_df.rename({"stadium": "name"})

# UPSERT stadium
con.execute("""
USE silver;
MERGE INTO stadium AS tgt
USING stadium_df AS src
  ON tgt.name = src.name
WHEN MATCHED THEN UPDATE SET
    run_id                       = src.run_id,
    pipeline_id                  = src.pipeline_id,
    silver_source_processed        = CURRENT_DATE,
WHEN NOT MATCHED THEN INSERT (
    name, country_id, run_id, pipeline_id, silver_source_processed,
    bronze_raw_date_ingested, bronze_landing_date_ingested, bronze_source
) VALUES (
    src.name, src.country_id, src.run_id, src.pipeline_id, src.silver_source_processed,
    src.bronze_raw_date_ingested, src.bronze_landing_date_ingested, src.bronze_source
);
""")

con.close()


In [ ]:
# team
# load the data from the silver parquet file
df = pl.read_parquet("data/silver_pre_nf.parquet")

# Surviorship and column selection
home_team = df.select(["home_team", "country", "run_id","pipeline_id","silver_source_processed","bronze_raw_date_ingested","bronze_landing_date_ingested","bronze_source"]) \
	.unique("home_team") \
	.rename({"home_team": "team"})
away_team = df.select(["away_team", "country","run_id","pipeline_id","silver_source_processed","bronze_raw_date_ingested","bronze_landing_date_ingested","bronze_source"]) \
	.unique("away_team") \
	.rename({"away_team": "team"})

teams = home_team.join(away_team, on="team", how="semi")

con = duckdb.connect("data/silver_3nf.duckdb")
# Resolve any foreign keys
country_lookup_df = con.execute("select country_id, country from silver.country").pl()

teams = teams.join(country_lookup_df, on="country", how="left")
teams = teams.drop("country")

# # UPSERT team
con.execute("""
USE silver;
MERGE INTO team AS tgt
USING teams AS src
  ON tgt.name = src.team
WHEN MATCHED THEN UPDATE SET
    run_id                       = src.run_id,
    pipeline_id                  = src.pipeline_id,
    silver_source_processed        = CURRENT_DATE
WHEN NOT MATCHED THEN INSERT (
    name, run_id, country_id, pipeline_id, silver_source_processed,
    bronze_raw_date_ingested, bronze_landing_date_ingested, bronze_source
) VALUES (
    src.team, src.country_id, src.run_id, src.pipeline_id, src.silver_source_processed,
    src.bronze_raw_date_ingested, src.bronze_landing_date_ingested, src.bronze_source
);
""")

con.close()

In [ ]:
# match
# load the data from the silver parquet file
df = pl.read_parquet("data/silver_pre_nf.parquet")

# Surviorship and column selection

con = duckdb.connect("data/silver_3nf.duckdb")

# Resolve any foreign keys
team_lookup_df = con.execute("select team_id, name from silver.team").pl()
df = df.join(team_lookup_df, right_on='name', left_on="home_team", how="left")	\
		.rename({"team_id": "home_team_id"})
df = df.join(team_lookup_df, right_on='name', left_on="away_team", how="left")	\
		.rename({"team_id": "away_team_id"})
stadium_lookup_df = con.execute("select stadium_id, name from silver.stadium").pl()
df = df.join(stadium_lookup_df, right_on='name', left_on='stadium', how="left")

# calculate hash
#based off match date, home_team, away_team
df = df.with_columns(
    pl.struct(['match_date','home_team','away_team']).hash(seed=0).alias("silver_integrated_hash")
)

df.drop(["home_team", "away_team", "competition", "stadium", "city", "country", "neutral", "world_cup"])


# UPSERT team
con.execute("""
USE silver;
MERGE INTO match AS tgt
USING df AS src
  ON tgt.silver_integrated_hash = src.silver_integrated_hash
WHEN MATCHED THEN UPDATE SET
	home_score				   		= src.home_score,
	away_score				   		= src.away_score,
    run_id                       	= src.run_id,
    pipeline_id                  	= src.pipeline_id,
    silver_source_processed        	= CURRENT_DATE

WHEN NOT MATCHED THEN INSERT (
    match_date, home_team_id, away_team_id, home_score, away_score, stadium_id, silver_integrated_hash,
    run_id, pipeline_id, silver_source_processed,bronze_raw_date_ingested, 
    bronze_landing_date_ingested, bronze_source
) VALUES (
    src.match_date, src.home_team_id, src.away_team_id, src.home_score, src.away_score, src.stadium_id,
    src.silver_integrated_hash, src.run_id, src.pipeline_id, src.silver_source_processed,
    src.bronze_raw_date_ingested, src.bronze_landing_date_ingested, src.bronze_source
);
""")

con.close()

In [ ]:

con.execute("""
USE silver;
DROP TABLE IF EXISTS match;
CREATE TABLE match (
        match_id    						INTEGER PRIMARY KEY DEFAULT nextval('seq_match_id'),
        match_date 							DATE NOT NULL, 
        home_team_id 						INTEGER NOT NULL,
        away_team_id  						INTEGER NOT NULL,
        home_score 							INT NOT NULL DEFAULT 0,
        away_score							INT NOT NULL DEFAULT 0,
        stadium_id							INTEGER NOT NULL,
        silver_integrated_hash				STRING NOT NULL,
        run_id								STRING NOT NULL,
        pipeline_id							STRING NOT NULL,
		silver_3nf_processed				DATE NOT NULL DEFAULT CURRENT_DATE,
        silver_source_processed 				DATE NOT NULL,
        bronze_raw_date_ingested 			DATE NOT NULL,
        bronze_landing_date_ingested		DATE NOT NULL,
		bronze_source						STRING NOT NULL,
        FOREIGN KEY (home_team_id) REFERENCES team(team_id),
        FOREIGN KEY (away_team_id) REFERENCES team(team_id),
        FOREIGN KEY (stadium_id) REFERENCES stadium(stadium_id),
    );
                   
""")

con.close()